# Step 3A.1 - Patch Runtime Baseline Report

This notebook does **not** rerun LLaDA decoding.

It reuses the existing Step 3A `dev_tune_200` outputs and recomputes the runtime baseline report with:

```text
primary_paraphrase_bucket = declarative_paraphrases
qa_format_generalization = secondary_diagnostic
constraint_filtering = enabled
analysis_500_used = false
```

Output:

```text
runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_v3/
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Preflight And Input Discovery

In [ ]:
import json
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert PROJECT_DIR.exists(), f"Missing project dir: {PROJECT_DIR}"

required_files = [
    "llada_experiment_reports.py",
    "runs/counterfact_direction1_v1/dev_tune_200_base_coverage/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_base_seed0/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json",
]
for name in required_files:
    path = PROJECT_DIR / name
    assert path.exists(), f"Missing required file: {path}"

summary_candidates = [
    "runs/counterfact_direction1_v1/dev_tune_200_base_coverage/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_base_seed0/summary.json",
]

split_target = [
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_logit_bias/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_support/summary.json",
]
combined_target = [
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target/summary.json",
]

if all((PROJECT_DIR / path).exists() for path in split_target):
    summary_candidates.extend(split_target)
elif all((PROJECT_DIR / path).exists() for path in combined_target):
    summary_candidates.extend(combined_target)
else:
    raise RuntimeError("Missing Step 3A target baseline summaries")

summary_candidates.extend([
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_bridge_controls/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_mc_bridge/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_raw_bridge_gated/summary.json",
])

summary_paths = []
seen = set()
for rel in summary_candidates:
    path = PROJECT_DIR / rel
    if not path.exists():
        raise RuntimeError(f"Missing Step 3A summary input: {path}")
    if path not in seen:
        summary_paths.append(path)
        seen.add(path)

try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()
except Exception as exc:
    commit = f"unavailable: {exc!r}"

print("Python:", sys.version)
print("git commit:", commit)
print("Target summaries:", "split" if all((PROJECT_DIR / path).exists() for path in split_target) else "combined")
print("Input summaries:")
for path in summary_paths:
    print(" ", path.relative_to(PROJECT_DIR))


## Overwrite Guard

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
out_dir = project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_v3"
if out_dir.exists():
    raise RuntimeError(
        f"Report directory already exists: {out_dir}. "
        "Delete it manually or choose a new report version. Do not overwrite silently."
    )
print("Overwrite guard passed:", out_dir)


## Recompute Report V3

In [ ]:
import subprocess
import sys
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
out_dir = project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_v3"
thresholds_path = project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json"

cmd = [
    sys.executable,
    "llada_experiment_reports.py",
    "--input",
    *[str(path.relative_to(project)) for path in summary_paths],
    "--output_dir",
    str(out_dir.relative_to(project)),
    "--thresholds_path",
    str(thresholds_path.relative_to(project)),
    "--bootstrap_baseline",
    "base",
    "--bootstrap_candidates",
    "target_logit_bias",
    "target_candidate_insert",
    "prompt_memory",
    "myopic_score",
    "no_rollout_bridge",
    "mc_bridge",
    "raw_bridge_gated_subject",
    "--bootstrap_metric",
    "exact_rate",
    "--bootstrap_samples",
    "2000",
    "--seed",
    "0",
]

print("Running:")
print(" ".join(cmd))
subprocess.run(cmd, cwd=project, check=True)
print("Wrote:", out_dir)


## Inspect Report V3

In [ ]:
import csv
import json
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
report_dir = project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report_v3"
thresholds_path = project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json"
expected = [
    "selection_scores.csv",
    "selection_scores_feasible.csv",
    "constraint_summary.csv",
    "aggregate_method_bucket.csv",
    "paired_bootstrap.csv",
    "report_summary.json",
]
for name in expected:
    path = report_dir / name
    assert path.exists(), f"Missing report artifact: {path}"

with thresholds_path.open() as f:
    thresholds = json.load(f)
expected_selfloc = float(thresholds["selfloc_base"])
assert 0.0 < expected_selfloc < 1.0, expected_selfloc

with (report_dir / "report_summary.json").open() as f:
    payload = json.load(f)
assert payload["primary_paraphrase_bucket"] == "declarative_paraphrases"
assert payload["qa_format_generalization"] == "secondary_diagnostic"
assert payload["constraint_filtering"] is True
assert payload["analysis_500_used"] is False
assert abs(float(payload["selfloc_base"]) - expected_selfloc) < 1e-8

with (report_dir / "selection_scores.csv").open(newline="") as f:
    scores = list(csv.DictReader(f))
with (report_dir / "selection_scores_feasible.csv").open(newline="") as f:
    feasible = list(csv.DictReader(f))
with (report_dir / "constraint_summary.csv").open(newline="") as f:
    constraints = list(csv.DictReader(f))

for row in scores:
    assert abs(float(row["selfloc_base"]) - expected_selfloc) < 1e-8, row

feasible_methods = {row["method"] for row in feasible}
assert "base" in feasible_methods, feasible_methods
assert "raw_bridge_gated_subject" in feasible_methods, feasible_methods

raw_bridge_rows = [row for row in feasible if row["method"] == "raw_bridge_gated_subject"]
assert raw_bridge_rows, "raw_bridge_gated_subject missing from feasible rows"
raw_score = float(raw_bridge_rows[0]["feasible_selection_score"])
assert raw_score > 0.25, raw_bridge_rows[0]

print("expected selfloc:", expected_selfloc)
print("selection rows:", len(scores))
print("feasible rows:", len(feasible))
print("constraint rows:", len(constraints))
print()
print("Selection scores:")
for row in scores:
    print(row)
print()
print("Feasible scores:")
for row in feasible:
    print(row)


## Expected Outcome

The patched report should make unsafe methods ineligible by setting:

```text
constraint_pass = false
feasible_selection_score = empty
```

The raw `selection_score` is still reported, but final dev selection should use `feasible_selection_score` after the gated Step 3B report is built.
